In [91]:
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

songs = pd.read_csv("./data/song_vectors.csv.gz")
vocab = pd.read_csv("./data/vocabulary.csv")
top_40 = pd.read_csv("./data/top_40_songs_by_genre.csv")

word_cols = vocab["word"].tolist()
meta_cols = [col for col in songs.columns if col not in word_cols]

X = songs[word_cols].to_numpy(dtype=float)
print(X)

[[ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ... 11.  0.  0.]
 ...
 [ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  3.  0.  0.]
 [ 0.  0.  0. ...  2.  0.  0.]]


In [92]:
# Term frequency: repeated uses of one word count less and less.
TF = np.log1p(X) # Compute TF

# Document frequency: how many songs use each word at least once.
N = X.shape[0] # Determine number of documents
df = (X > 0).sum(axis=0) # Compute document frequency

    
IDF = np.log(N / df) # Compute IDF


# IDF is one number per word, so NumPy broadcasts it across every song row.
TFIDF = TF * IDF

print(pd.DataFrame({
    "word": word_cols,
    "document_frequency": df,
    "idf": IDF,
}).sort_values("idf", ascending=True).head(10))

print( pd.DataFrame({
    "word": word_cols,
    "document_frequency": df,
    "idf": IDF,
}).sort_values("idf", ascending=False).head(10) )

     word  document_frequency       idf
324     i                5966  0.293365
22    and                5960  0.294371
795   you                5671  0.344076
409    me                5420  0.389346
331    in                5345  0.403280
334    is                5191  0.432515
336    it                5045  0.461044
466   not                4961  0.477834
475    of                4785  0.513956
662  that                4597  0.554038
     word  document_frequency       idf
176   doo                  26  5.729100
258  funk                  48  5.115996
726   ven                  56  4.961845
327   ihr                  59  4.909659
770   wir                  60  4.892852
201    er                  63  4.844062
711     u                  66  4.797542
162  dich                  68  4.767689
596   sie                  78  4.630488
208    eu                  81  4.592748


In [111]:
most_frequent = pd.DataFrame({
    "word": word_cols,
    "document_frequency": df,
    "idf": IDF,
})

# Most frequent words are those with a frequency >= 4500
most_frequent_4500 = most_frequent[most_frequent['document_frequency'] >= 4500]
list_most_frequent_4500 = most_frequent_4500['word'].tolist()

# Most frequent words are those with a frequency >= 4000
most_frequent_4000 = most_frequent[most_frequent['document_frequency'] >= 4000]
list_most_frequent_4000 = most_frequent_4000['word'].tolist()
# print(list_most_frequent_3000)


# Most frequent words are those with a frequency >= 3000
most_frequent_3000 = most_frequent[most_frequent['document_frequency'] >= 3000]
list_most_frequent_3000 = most_frequent_3000['word'].tolist()
# print(list_most_frequent_3000)



# Most frequent words are those with a frequency >= 1000
most_frequent_1000 = most_frequent[most_frequent['document_frequency'] >= 1000]
list_most_frequent_1000 = most_frequent_1000['word'].tolist()
# print(list_most_frequent_1000)



# Most frequent words are those with a frequency >= 750
most_frequent_750 = most_frequent[most_frequent['document_frequency'] >= 750]
list_most_frequent_750 = most_frequent_750['word'].tolist()
print(list_most_frequent_750)

# Most frequent words are those with a frequency >= 750
most_frequent_500 = most_frequent[most_frequent['document_frequency'] >= 500]
list_most_frequent_500 = most_frequent_500['word'].tolist()
print(list_most_frequent_500)

['about', 'again', 'all', 'alway', 'am', 'an', 'and', 'are', 'around', 'as', 'at', 'away', 'babi', 'back', 'be', 'been', 'befor', 'better', 'but', 'by', 'ca', 'call', 'can', 'caus', 'come', 'could', 'day', 'de', 'did', 'die', 'do', 'down', 'dream', 'en', 'end', 'even', 'ever', 'everi', 'eye', 'face', 'fall', 'feel', 'find', 'for', 'from', 'get', 'girl', 'give', 'go', 'gone', 'gonna', 'good', 'got', 'had', 'hand', 'hard', 'has', 'have', 'he', 'head', 'hear', 'heart', 'her', 'here', 'his', 'hold', 'home', 'how', 'i', 'if', 'in', 'into', 'is', 'it', 'just', 'keep', 'know', 'la', 'leav', 'let', 'life', 'light', 'like', 'littl', 'live', 'long', 'look', 'love', 'make', 'man', 'me', 'mind', 'more', 'much', 'my', 'need', 'never', 'new', 'night', 'no', 'not', 'noth', 'now', 'of', 'off', 'oh', 'on', 'one', 'onli', 'or', 'our', 'out', 'over', 'place', 'que', 'right', 'run', 'said', 'say', 'see', 'she', 'show', 'so', 'some', 'start', 'stay', 'still', 'take', 'tell', 'that', 'them', 'then', 'there'

In [94]:
def top_matches(scores, query_track_id, k=10, query_artist_id=None):
    results = songs[meta_cols].copy()
    results["score"] = scores

    results = results[results["track_id"] != query_track_id]
    if query_artist_id is not None:
        results = results[results["artist_id"] != query_artist_id]

    return results.sort_values("score", ascending=False).head(k)

In [95]:
def cosine_scores(matrix, query_vector):
    numerators = matrix @ query_vector
    denominators = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vector)

    return np.divide(
        numerators,
        denominators,
        out=np.zeros(len(numerators)),
        where=denominators != 0,
    )

In [112]:
songs = pd.read_csv("./data/song_vectors.csv.gz")
#print(songs)
vocab = pd.read_csv("./data/vocabulary.csv")
top_40 = pd.read_csv("./data/top_40_songs_by_genre.csv")

word_cols = vocab["word"].tolist()
meta_cols = [col for col in songs.columns if col not in word_cols]

# Remove words with a frequency >= 4500
word_cols_X00 = [word for word in word_cols if word not in list_most_frequent_4500]
# Remove words with a frequency >= 4000
word_cols_X0 = [word for word in word_cols if word not in list_most_frequent_4000]
# Remove words with a frequency >= 3000
word_cols_X1 = [word for word in word_cols if word not in list_most_frequent_3000]
# Remove words with a frequency >= 1000
word_cols_X2 = [word for word in word_cols if word not in list_most_frequent_1000]
# Remove words with a frequency >= 750
word_cols_X3 = [word for word in word_cols if word not in list_most_frequent_750]
# Remove words with a frequency >= 500
word_cols_X4 = [word for word in word_cols if word not in list_most_frequent_500]

X00 = songs[word_cols_X00].to_numpy(dtype=float)

X0 = songs[word_cols_X0].to_numpy(dtype=float)

X1 = songs[word_cols_X1].to_numpy(dtype=float)

X2 = songs[word_cols_X2].to_numpy(dtype=float)

X3 = songs[word_cols_X3].to_numpy(dtype=float)

X4 = songs[word_cols_X4].to_numpy(dtype=float)


print(len(list_most_frequent_3000))
print(len(word_cols))


28
800


In [97]:
songs = pd.read_csv("./data/song_vectors.csv.gz")
vocab = pd.read_csv("./data/vocabulary.csv")
top_40 = pd.read_csv("./data/top_40_songs_by_genre.csv")


word_cols = vocab["word"].tolist()
meta_cols = [col for col in songs.columns if col not in word_cols]


def top_40_query(top_40=top_40, X=X1, strat="dot", recommendations=3):
    hits = pd.DataFrame(columns=["type", "song_title", "genre", "recommendations_genre", "num_recommendations", "hit_count"])
        # Term frequency: repeated uses of one word count less and less.
    TF = np.log1p(X) # Compute TF

    # Document frequency: how many songs use each word at least once.
    N = X.shape[0] # Determine number of documents
    df = (X > 0).sum(axis=0) # Compute document frequency
    IDF = np.log(N / df) # Compute IDF

    # IDF is one number per word, so NumPy broadcasts it across every song row.
    TFIDF = TF * IDF
    
    for index, row in top_40.iterrows():
        #print("Row:", row)
        query_row = index
        query_track_id = top_40.loc[query_row, "track_id"]
        songs_index = songs.index[songs["track_id"] == query_track_id].to_list()[0]
        query_track_name = top_40.loc[query_row, "song_title"]
        query_track_genre = top_40.loc[query_row, "genre"]
        query = X[songs_index]
        query_tfidf = TFIDF[songs_index]
        tracker = []

        #print("Query:", query)

        #songs.loc[[query_row], ["artist_name", "song_title", "genre", "n_words", "nnz_words"]]

        if strat == "dot":
            dot_scores = X @ query
            matches = top_matches(dot_scores, query_track_id, k=recommendations)
            #print("Matches:", matches)

        elif strat == "cosine":
            cos_scores = cosine_scores(X, query)
            matches = top_matches(cos_scores, query_track_id, k=recommendations)
            #print("Matches:", matches)

        elif strat == "tfidf_cosine":
            tfidf_cos_scores = cosine_scores(TFIDF, query_tfidf)
            matches = top_matches(tfidf_cos_scores, query_track_id, k=recommendations)
            #print("Matches:", matches)

        for index, match in matches.iterrows():
            # print("Match genre:" + match["genre"])
            # print(type(match["genre"]))
            # print("Target genre:" + query_track_genre)
            # print(type(query_track_genre))

            if match["genre"] == query_track_genre:
                #print("hit")
                tracker.append(True)
            else:
                #print("miss")
                tracker.append(False)

            #print(tracker)

        rec_genres = matches["genre"].tolist()
        hits.loc[len(hits)] = [strat, query_track_name, query_track_genre, rec_genres, recommendations, tracker.count(True)]

    return hits
        

In [113]:
test_with_X00 = top_40_query(top_40 = top_40, X = X00,  strat="tfidf_cosine", recommendations=3)
test_with_X00['hit_count'].value_counts()

hit_count
0    314
1    154
2     61
3     31
Name: count, dtype: int64

In [109]:
test_with_X0 = top_40_query(top_40 = top_40, X = X0,  strat="tfidf_cosine", recommendations=3)
test_with_X0['hit_count'].value_counts()

hit_count
0    314
1    153
2     62
3     31
Name: count, dtype: int64

In [98]:
test_with_X1 = top_40_query(top_40 = top_40, X = X1,  strat="tfidf_cosine", recommendations=3)
test_with_X1['hit_count'].value_counts()

hit_count
0    319
1    150
2     60
3     31
Name: count, dtype: int64

In [99]:
test_with_X2 = top_40_query(top_40 = top_40, X = X2,  strat="tfidf_cosine", recommendations=3)
test_with_X2['hit_count'].value_counts()

hit_count
0    327
1    148
2     58
3     27
Name: count, dtype: int64

In [100]:
test_with_X3 = top_40_query(top_40 = top_40, X = X3,  strat="tfidf_cosine", recommendations=3)
test_with_X3['hit_count'].value_counts()

hit_count
0    335
1    148
2     54
3     23
Name: count, dtype: int64

In [101]:
test_with_X4 = top_40_query(top_40 = top_40, X = X4,  strat="tfidf_cosine", recommendations=3)
test_with_X4['hit_count'].value_counts()

hit_count
0    341
1    142
2     50
3     27
Name: count, dtype: int64

In [114]:
test_with_X00_mean = test_with_X00["hit_count"].sum()/(test_with_X00.shape[0]*test_with_X00["num_recommendations"][0])
test_with_X00_mean

np.float64(0.21964285714285714)

In [110]:
test_with_X0_mean = test_with_X0["hit_count"].sum()/(test_with_X0.shape[0]*test_with_X0["num_recommendations"][0])
test_with_X0_mean

np.float64(0.22023809523809523)

In [103]:
test_with_X1_mean = test_with_X1["hit_count"].sum()/(test_with_X1.shape[0]*test_with_X1["num_recommendations"][0])
test_with_X1_mean

np.float64(0.21607142857142858)

In [104]:
test_with_X2_mean = test_with_X2["hit_count"].sum()/(test_with_X2.shape[0]*test_with_X2["num_recommendations"][0])
test_with_X2_mean

np.float64(0.20535714285714285)

In [105]:
test_with_X3_mean = test_with_X3["hit_count"].sum()/(test_with_X3.shape[0]*test_with_X3["num_recommendations"][0])
test_with_X3_mean

np.float64(0.19345238095238096)

In [106]:
test_with_X4_mean = test_with_X4["hit_count"].sum()/(test_with_X4.shape[0]*test_with_X4["num_recommendations"][0])
test_with_X4_mean

np.float64(0.19226190476190477)